**Flood Mapping & Impact Assessment**

In [ ]:
# import libraries
import ee
import geemap
import pandas as pd
import geopandas as gpd

import os

## . Connect to GEE

In [ ]:
# Authenticate GEE
ee.Authenticate()

#Initialize GEE
ee.Initialize()

True

## . Flood Parameters

In [ ]:
# Dates during floods
after_start = "2022-09-14"
after_end = "2022-11-18"

before_start = "2022-08-01"
before_end   = "2022-09-10"

flood_threshold = -15
speckle_radius = 50

## . Visualization Parameters

In [ ]:
# Visualization Parameters for Boundary
vis_params_aoi = ...


# Visualization Parameters for Sentinel-2
vis_params_s2_rgb = ...

vis_params_s2_fcc = ...

# Visualization Parameters for Sentinel-1
vis_params_s1_vv = ....

## . Helper Functions

## . Area of Interest (AOI)

In [5]:
# AOI (GeoJSON)
aoi = ee.Geometry.Polygon(
    [[[6.893233,7.89773],
      [6.666644,7.89773],
      [6.666644,7.683773],
      [6.893233,7.683773],
      [6.893233,7.89773]]]
)

# FAO GAUL Boundaries (Nigeria & Lokoja)

fao_l2 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2")

nga_lga = fao_l2.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))

lokoja = nga_lga.filter(ee.Filter.eq("ADM2_NAME", "Lokoja"))

## . Download and Pre-Process Sentinel-2

In [ ]:
# Function to download Sentinel-2
def download_s2(start_date: str, end_date : str, cloud_perc: int, extent : ee.Feature | ee.Geometry):
    """
    Download Sentinel-2 images
    # COMPLETE doc-string
    """
    img_collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                    ... # Dates filter
                    ... # Cloud filter
                    ... # Boundary filter
                    )   

    img_count =  ... # Number of image in the collection
    print(f"There are {img_count} images in the Sentinel-2 image collection\n")
    
    return img_collection



# Function to apply cloud masking to Sentinel-2
# See reference: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_CLOUD_PROBABILITY
def mask_clouds_s2(image: ee.Image, cloud_thresh: int):
    """
    Apply cloud masking using cloud probability
    # COMPLETE DOCTSRING
    """
    clear_threshold = cloud_thresh
    clouds = ee.Image(
            image.get("cloud_mask")
            ).select("probability")
    
    no_clouds = clouds.lt(clear_threshold)
    masked_image = image.updateMask(no_clouds).copyProperties(image, ["system:time_start"])
    
    return masked_image
    

# Masks from the 20m and 60m bands for bad data at scene edges
def mask_edges_s2(image : ee.Image):
    """
    ....
    """
    masked_image = image.updateMask(
                    image.select("B8A").mask().updateMask(image.select("B9").mask())
                    ).copyProperties(image, ["system:time_start"])
    
    return masked_image 


# Function to convert Sentinel-2 DN to reflectance
def scale_bands_s2(image: ee.Image):
        """
        """
        scaled_image = ... # Multiply image by 0.0001 and copy "system:time_start" property 
        return scaled_image


# Function to resample bands in Sentinel-2 to 10m
def resample_bands_s2(image: ee.Image):
        """
        """
        bands_10m = image.select([...]) # Select 10m bands
        bands_20m = image.select([...]) # Select 20m bands
        resmapled_image = ....resample("bilinear").reproject(
                                                            crs = bands_10m.projection(), 
                                                             scale=10)
        
        combined_10m = bands_10m.addBands(...) # Combine original 10m with 20m
        
        return combined_10m.copyProperties(image, ["system:time_start"])


# Function to compute spectral indices
def compute_indices_s2(image:ee.Image):
        """
        .....
        """
        
        ndvi = image.normalizedDifference(["B8", "B4"]).rename("ndvi")
        ndwi = image.normalizedDifference(["B3", "B8"]).rename("ndwi")
        mndwi = image.normalizedDifference(["B3", "B11"]).rename("mndwi")
        evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            "NIR":image.select("B8"),
            "RED": image.select("B4"),
            "BLUE": image.select("B2")
        }
        ).rename("evi")
        
        updated_image = image.addBands([ndvi, ndwi, mndwi, evi]).copyProperties(image, ["system:time_start"])
        
        return updated_image 



In [ ]:
# Download Sentinel-2 images
s2_img_col = download_s2(after_start, cloud_perc: int, extent : ee.Feature | ee.Geometry)

# Apply cloud masking to ALL IMAGES in the Sentinel-2 collection
...

# Resample bands for ALL IMAGES in the Sentinel-2 collection
...

# Rescale pixel values  for ALL IMAGES in the Sentinel-2 collection
...

# Compute Indices for ALL IMAGES in the Sentinel-2 collection
...

In [ ]:
# Convert Sentinel-2 images to a List
s2_img_list = 

In [ ]:
# Visualize Sentinel-2 imagery


## . Download and Pre-process Sentinel-1

In [ ]:
# Function to download Sentinel-1
def download_s1(start_date: str, end_date: str, extent: ee.Feature | ee.Geometry):
    """
    """
    img_collection = (ee.ImageCollection("COPERNICUS/S1_GRD")
                        ... # Dates filter
                        ... # InstrumentMode
                        ... # Resolution filter
                        ... # Boundary filter
                         .select(["VV", "VH"]))

    # Number of images in the Sentinel-1 Collection
    img_count = ...

    print(f"There are {img_count} Sentinel-1 Images")


# Function to apply Speckle filter to Sentinel-1 (focal median)
def focal_speckle_filter(image : ee.Image, radius : int,  unit: str = "meters"):
    """
    Focal Median filter with circle kernel.
    """
    image_filtered = ....focal_median(radius, "circle", unit)
    image_filtered.copyProperties(image, image.propertyNames())

    return image_filtered

In [ ]:
# Download Sentinel-1 imagery
s1_img_col = ...

# Apply Speckle filter to ALL IMAGES in the Sentinel-1 collection
...


In [ ]:
# Convert Sentinel-1 images to a List
s1_img_list = 

In [ ]:
# Visualize Sentinel-1 imagery

In [ ]:
s1_img_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
                .filterDate(start_date, end_date)
                .filter(ee.Filter.eq("instrumentMode", "IW"))
                .filterMetadata("resolution_meters", "equals", 10)
                .filterBounds(aoi)
                .select(["VV", "VH"]))

# image.focal_median(radius, "circle", "meters")

## . Download Landsat-8

In [ ]:
def download_l8(start_date: str, end_date: str, extent: ee.Feature | ee.Geometry):
        """
        """
    
        # download and Filter and Landsat-8 image collection 
        img_collection = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                                  ... # Dates filter
                                  ... # Cloud filter
                                  ... # Boundary filter
                    )   

        img_count =  ... # Number of image in the collection

        print(f"There are {img_count} images in the Landsat-8 image collection\n")

        return img_collection 


# Band scaling factors for Landsat-8
def scale_factors_l8(image : ee.Image):
        """
        """

        optical_bands = image.select(...).multiply(...).add(...)  # scaling of reflectance values
        thermal_bands = image.select(...).multiply(...).add(...) # scaling of temp/thermal values

        scaled_bands =  image.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)
        
        return scaled_bands


# Cloud Masking for Landsat-8
def mask_l8_sr(image : ee.Image):

        qa_mask = image.select("QA_PIXEL").bitwiseAnd(int("11111", 2)).eq(0)
        masked_image = image.updateMask(qa_mask).copyProperties(image, ["system:time_start"])


        return masked_image

# Function to compute spectral indices
def compute_indices_l8(image:ee.Image):
        """
        """
        .... # Use the example in Sentinel-2 above to complete this

In [ ]:
# Download Landsat-8 images
l8_img_col = download_l8(after_start, cloud_perc: int, extent : ee.Feature | ee.Geometry)

# Apply cloud masking to ALL IMAGES in the Landsat-8 collection
...


# Rescale pixel values  for ALL IMAGES in the Landsat-8 collection
...

# Compute Indices for ALL IMAGES in the SLandsat-8 collection
...

## . Download Building Dataset

In [8]:
# Load Building Dataset (Google Open Buildings)
buildings = ee.FeatureCollection(
    "GOOGLE/Research/open-buildings-temporal/v1"
)

# Clip buildings to AOI
buildings_aoi = buildings.filterBounds(aoi)

Visualization

In [11]:
import geemap

# Create map
Map = geemap.Map(center=[7.8175, 6.7730], zoom=6)

# Add Nigeria boundaries
Map.addLayer(nga_bdry_l0, {"color": "black"}, "NGA Bdry L0")
Map.addLayer(nga_bdry_l1, {"color": "blue"}, "NGA State")
Map.addLayer(nga_bdry_l2, {"color": "green"}, "NGA LGA L2")

# Add Lokoja
Map.addLayer(lokoja_by_name, {"color": "black"}, "Lokoja (Name)")
Map.addLayer(lokoja_by_code, {"color": "red"}, "Lokoja (Code)")

# Add AOI boundary
Map.addLayer(aoi, {"color": "yellow"}, "AOI")

# Display map
Map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## . Flood Mapping

## . Damage / Impact Assessment